# All Star Scoring Analysis — 2025–26 Season

## Analytical Objective

Determine which scoring components differentiate successful competitive All Star
cheerleading performances across Levels 1, 2, 3, 4, 4.2, 5, and 6 while
separating routine construction, performance errors, level-specific competitive
context, and scoring-format differences.

## Analytical Grain

The primary observation is one team performance within a competition, level,
division, and round.

Primary outcomes:

- Performance score
- Placement within competition / level / division / round

Event score is retained as source data but is not treated as a universally
comparable outcome because event aggregation rules vary by competition.

Raw score is also retained but is not treated as directly comparable across all
performances because Level 1 and Level 2 Mini use a 46-point no-toss scorecard,
while the standard scorecard totals 50 points.

## Analysis Lenses

1. **All performances**
   - Measures actual competitive outcomes, including deductions, missed skills,
     and other competitive errors.

2. **Zero-deduction performances**
   - Separates performances with no recorded deductions to reduce the influence
     of falls and other penalized performance errors.
   - Zero deductions does not necessarily indicate that every intended or
     compulsory skill was successfully performed. Omitted or uncredited skills
     may reduce category difficulty scores without producing a deduction.

3. **Deduction analysis**
   - Examines deductions separately to quantify the competitive impact of
     performance errors.

4. **Level and competitive-context analysis**
   - Evaluates whether scoring relationships are consistent across competitive
     levels.
   - Comparisons across levels will control for level and competitive context
     rather than assuming that identical numeric category scores represent
     identical athletic difficulty.

## Scoring Context

Most objective difficulty categories use the same nominal scoring ranges across
Levels 1–6 and Level 4.2. However, the skills required to earn those scores
differ substantially by competitive level.

Many difficulty categories are effectively compulsory within a level: teams
know the skills required to maximize these categories and generally construct
routines to meet those requirements.

These include:

- Stunt difficulty
- Stunt DOD
- Stunt MAX / MPD
- Toss difficulty, when applicable
- Standing tumbling difficulty
- Standing tumbling DOD
- Running tumbling difficulty
- Running tumbling DOD
- Running tumbling MAX / MPD
- Jump difficulty

A missed or dropped skill can simultaneously produce a deduction and cause the
skill to receive reduced or no difficulty credit. Therefore, difficulty scores
from performances with deductions may reflect execution failure rather than
routine-construction decisions.

Pyramid difficulty and dance difficulty differ from the objective/max-out
difficulty categories because their difficulty scores are subjectively
evaluated. They will therefore be analyzed separately.

Low variance in an objective difficulty category among zero-deduction
performances does not necessarily mean the category is unimportant. It may
instead indicate that competitive teams routinely construct routines to
maximize the available score.

### Toss Applicability

Level 1 and Level 2 Mini divisions do not have a toss category and use a
46-point scorecard. Toss scores are therefore structurally missing for these
performances and are not interpreted as zero scores or missing-data errors.

All other included divisions use the standard 50-point scorecard with a toss
category.

### Category Scale Interpretation

Raw category point ranges are not directly comparable measures of judging
variability. Execution categories use technical scoring ranges in which scores
begin at a maximum and are reduced in similar increments for identified
execution issues, despite categories having different nominal maximum values.

Accordingly, category importance will not be inferred from raw score range or
percentage of nominal maximum alone. Comparative analyses will use
within-category and competitive-context standardization where appropriate while
preserving the different scoring mechanisms represented by each rubric.

## Competition Format Considerations

Competition scoring formats are not uniform.

- Many two-day competitions use approximately 25% preliminary / 75% final
  weighting.
- CHEERSPORT weights the team's higher-scoring day at 75%.
- Summit advancement rounds do not use score carryover.

For this reason, individual round performance is the primary analytical unit.

Competition-specific event aggregation will be handled separately when needed.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = Path(
    "../data/processed/"
    "season_2026_all_levels_performances.csv"
)

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "competition_id": "string",
        "division_id": "string",
        "team_id": "string",
        "level": "string",
    },
)

LEVEL_ORDER = [
    "1",
    "2",
    "3",
    "4",
    "4.2",
    "5",
    "6",
]

print(f"Performances: {len(df):,}")
print(f"Competitions: {df['competition_id'].nunique():,}")
print(f"Teams: {df['team_id'].nunique():,}")
print(f"Divisions: {df['division_id'].nunique():,}")

print()
print("Performances by level:")
print(
    df["level"]
    .value_counts()
    .reindex(LEVEL_ORDER)
    .to_string()
)

print()
print("Rounds:")
print(df["round"].value_counts().to_string())

print()
print("Scoring schemas:")
print(df["scoring_schema"].value_counts().to_string())

Performances: 31,151
Competitions: 160
Teams: 4,408
Divisions: 294

Performances by level:
level
1      7605
2      7599
3      6517
4      4131
4.2    1448
5      2362
6      1489

Rounds:
round
Finals        13196
Semifinals     5922
Prelims        5520
Round 1        3590
Round 2        2127
Wild Card       796

Scoring schemas:
scoring_schema
standard_50    23232
no_toss_46      7919


In [4]:
analysis_df = df.copy()

analysis_df["zero_deduction"] = (
    analysis_df["deductions"] == 0
)

deduction_summary = (
    analysis_df["zero_deduction"]
    .value_counts()
    .rename(index={
        True: "Zero deductions",
        False: "Recorded deduction",
    })
    .to_frame("performances")
)

deduction_summary["pct"] = (
    deduction_summary["performances"]
    / len(analysis_df)
    * 100
)

print(deduction_summary)

print("\nZero-deduction rate by level:")
print(
    analysis_df
    .groupby("level")["zero_deduction"]
    .agg(
        performances="size",
        zero_deduction_performances="sum",
        zero_deduction_rate="mean",
    )
    .reindex(LEVEL_ORDER)
    .assign(
        zero_deduction_rate=lambda x:
            x["zero_deduction_rate"] * 100
    )
    .to_string()
)

print("\nDeduction amounts:")
print(
    analysis_df["deductions"]
    .value_counts()
    .sort_index()
    .to_string()
)

                    performances     pct
zero_deduction                          
Zero deductions            16459 52.8362
Recorded deduction         14692 47.1638

Zero-deduction rate by level:
       performances  zero_deduction_performances  zero_deduction_rate
level                                                                
1              7605                         4780              62.8534
2              7599                         4468              58.7972
3              6517                         3369              51.6956
4              4131                         1857              44.9528
4.2            1448                          600              41.4365
5              2362                          969              41.0246
6              1489                          416              27.9382

Deduction amounts:
deductions
0.0000    16459
0.0100        1
0.0500       93
0.1000      667
0.1500      641
0.2000       86
0.2500     3563
0.3000      101
0.3500      174


In [5]:
compulsory_cols = [
    "stunt_difficulty",
    "stunt_dod",
    "stunt_max",
    "toss_difficulty",
    "standing_tumbling_difficulty",
    "standing_tumbling_dod",
    "running_tumbling_difficulty",
    "running_tumbling_dod",
    "running_tumbling_max",
    "jump_difficulty",
]

zero_deduction_df = analysis_df[
    analysis_df["zero_deduction"]
].copy()

ceiling_summary = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_df[
        zero_deduction_df["level"] == level
    ]

    for col in compulsory_cols:
        applicable = level_df[col].dropna()

        # Structural N/A categories, such as toss for Level 1,
        # are excluded rather than treated as zero scores.
        if applicable.empty:
            continue

        observed_max = applicable.max()
        at_max = (applicable == observed_max).sum()

        ceiling_summary.append({
            "level": level,
            "category": col,
            "performances": len(applicable),
            "observed_max": observed_max,
            "median": applicable.median(),
            "at_max": at_max,
            "pct_at_max": at_max / len(applicable) * 100,
            "std_dev": applicable.std(),
        })

ceiling_summary = pd.DataFrame(
    ceiling_summary
)

print("OBJECTIVE DIFFICULTY CEILING RATES BY LEVEL")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        ceiling_summary[
            ceiling_summary["level"] == level
        ]
        .sort_values(
            "pct_at_max",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "performances",
                "observed_max",
                "median",
                "pct_at_max",
                "std_dev",
            ],
        )
    )

OBJECTIVE DIFFICULTY CEILING RATES BY LEVEL


LEVEL 1
                    category  performances  observed_max  median  pct_at_max  std_dev
 running_tumbling_difficulty          4780        3.0000  3.0000     99.9791   0.0072
standing_tumbling_difficulty          4780        3.0000  3.0000     99.8954   0.0289
            stunt_difficulty          4780        4.5000  4.5000     99.7280   0.0613
             jump_difficulty          4780        2.0000  2.0000     99.2050   0.0695
        running_tumbling_max          4780        0.5000  0.5000     98.9121   0.0373
        running_tumbling_dod          4780        0.5000  0.5000     98.6192   0.0299
       standing_tumbling_dod          4780        1.0000  1.0000     97.7615   0.0639
                   stunt_max          4780        0.7000  0.7000     95.9623   0.0461
                   stunt_dod          4780        0.8000  0.8000     90.7741   0.0801

LEVEL 2
                    category  performances  observed_max  median  pct_at_max 

In [6]:
subjective_difficulty_cols = [
    "pyramid_difficulty",
    "dance_difficulty",
]

subjective_summary = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_df[
        zero_deduction_df["level"] == level
    ]

    for col in subjective_difficulty_cols:
        scores = level_df[col].dropna()

        observed_max = scores.max()
        at_max = (scores == observed_max).sum()

        subjective_summary.append({
            "level": level,
            "category": col,
            "performances": len(scores),
            "observed_max": observed_max,
            "min": scores.min(),
            "median": scores.median(),
            "pct_at_max": at_max / len(scores) * 100,
            "std_dev": scores.std(),
            "unique_scores": scores.nunique(),
        })

subjective_summary = pd.DataFrame(
    subjective_summary
)

print("SUBJECTIVE DIFFICULTY BY LEVEL")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        subjective_summary[
            subjective_summary["level"] == level
        ]
        .to_string(
            index=False,
            columns=[
                "category",
                "performances",
                "observed_max",
                "min",
                "median",
                "pct_at_max",
                "std_dev",
                "unique_scores",
            ],
        )
    )

SUBJECTIVE DIFFICULTY BY LEVEL


LEVEL 1
          category  performances  observed_max    min  median  pct_at_max  std_dev  unique_scores
pyramid_difficulty          4780        4.0000 2.5000  3.8000      8.0126   0.1277             13
  dance_difficulty          4780        1.0000 0.5000  0.9000     27.3849   0.1082              6

LEVEL 2
          category  performances  observed_max    min  median  pct_at_max  std_dev  unique_scores
pyramid_difficulty          4468        4.0000 2.0000  3.8000      8.0125   0.1217             13
  dance_difficulty          4468        1.0000 0.6000  0.9000     34.5568   0.1032              5

LEVEL 3
          category  performances  observed_max    min  median  pct_at_max  std_dev  unique_scores
pyramid_difficulty          3369        4.0000 3.2000  3.8000     11.1012   0.1073              7
  dance_difficulty          3369        1.0000 0.5000  0.9000     34.1644   0.0979              6

LEVEL 4
          category  performances  observed_max    

In [7]:
execution_cols = [
    "stunt_execution",
    "pyramid_execution",
    "toss_execution",
    "standing_tumbling_execution",
    "running_tumbling_execution",
    "jump_execution",
    "rc",
    "formations_transitions",
    "dance_execution",
    "show",
]

execution_summary = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_df[
        zero_deduction_df["level"] == level
    ]

    for col in execution_cols:
        scores = level_df[col].dropna()

        # Excludes structurally non-applicable categories,
        # such as toss execution for Level 1.
        if scores.empty:
            continue

        execution_summary.append({
            "level": level,
            "category": col,
            "performances": len(scores),
            "min": scores.min(),
            "median": scores.median(),
            "max": scores.max(),
            "mean": scores.mean(),
            "std_dev": scores.std(),
            "unique_scores": scores.nunique(),
        })

execution_summary = pd.DataFrame(
    execution_summary
)

print("EXECUTION AND PERFORMANCE CATEGORIES BY LEVEL")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        execution_summary[
            execution_summary["level"] == level
        ]
        .sort_values(
            "std_dev",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "performances",
                "min",
                "median",
                "max",
                "mean",
                "std_dev",
                "unique_scores",
            ],
        )
    )

EXECUTION AND PERFORMANCE CATEGORIES BY LEVEL


LEVEL 1
                   category  performances    min  median    max   mean  std_dev  unique_scores
            stunt_execution          4780 2.8000  3.8000 4.0000 3.7872   0.1304             11
          pyramid_execution          4780 3.2000  3.8000 4.0000 3.8047   0.1235              9
 running_tumbling_execution          4780 3.0000  3.8000 4.0000 3.8321   0.1208             11
standing_tumbling_execution          4780 3.1000  3.9000 4.0000 3.8618   0.1137             10
             jump_execution          4780 1.3000  1.8000 2.0000 1.8017   0.1097              8
            dance_execution          4780 0.5000  0.8000 1.0000 0.8082   0.1010              6
     formations_transitions          4780 1.3000  2.0000 2.0000 1.9413   0.0902              8
                       show          4780 1.4300  1.7700 2.0000 1.7806   0.0782             18
                         rc          4780 1.5300  1.8000 2.0000 1.7940   0.0778          

In [8]:
group_cols = [
    "competition_id",
    "level",
    "division_id",
    "round",
]

analysis_df["field_size"] = (
    analysis_df
    .groupby(group_cols)["team_id"]
    .transform("count")
)

analysis_df["placement_pct"] = (
    (analysis_df["rank"] - 1)
    / (analysis_df["field_size"] - 1)
)

# A one-team division has no meaningful relative placement.
analysis_df.loc[
    analysis_df["field_size"] == 1,
    "placement_pct"
] = np.nan

print(
    analysis_df[
        [
            "competition_id",
            "division_id",
            "round",
            "rank",
            "field_size",
            "placement_pct",
        ]
    ]
    .sort_values(
        ["field_size", "rank"],
        ascending=[False, True],
    )
    .head(25)
    .to_string(index=False)
)

print()
print("Field-size summary:")
print(
    analysis_df["field_size"]
    .describe()
    .to_string()
)

print()
print(
    "Single-team performances:",
    (analysis_df["field_size"] == 1).sum(),
)

competition_id     division_id      round  rank  field_size  placement_pct
      14478856 l3_junior_small Semifinals     1          32         0.0000
      14478856 l3_junior_small Semifinals     2          32         0.0323
      14478856 l3_junior_small Semifinals     3          32         0.0645
      14478856 l3_junior_small Semifinals     4          32         0.0968
      14478856 l3_junior_small Semifinals     5          32         0.1290
      14478856 l3_junior_small Semifinals     6          32         0.1613
      14478856 l3_junior_small Semifinals     6          32         0.1613
      14478856 l3_junior_small Semifinals     8          32         0.2258
      14478856 l3_junior_small Semifinals     9          32         0.2581
      14478856 l3_junior_small Semifinals    10          32         0.2903
      14478856 l3_junior_small Semifinals    11          32         0.3226
      14478856 l3_junior_small Semifinals    12          32         0.3548
      14478856 l3_junior_

In [9]:
analysis_df["is_winner"] = (
    analysis_df["rank"] == 1
)

analysis_df["top_3"] = (
    analysis_df["rank"] <= 3
)

analysis_df["top_quartile"] = (
    analysis_df["placement_pct"] <= 0.25
)

group_sizes = (
    analysis_df[
        group_cols + ["field_size"]
    ]
    .drop_duplicates()
)

print("COMPETITIVE GROUPS")
print()

print(f"Total groups: {len(group_sizes):,}")
print(
    "Single-team groups:",
    f"{(group_sizes['field_size'] == 1).sum():,}",
)
print(
    "Groups with 2 teams:",
    f"{(group_sizes['field_size'] == 2).sum():,}",
)
print(
    "Groups with 3+ teams:",
    f"{(group_sizes['field_size'] >= 3).sum():,}",
)

print()
print("GROUPS WITH 3+ TEAMS BY LEVEL")

print(
    group_sizes[
        group_sizes["field_size"] >= 3
    ]
    .groupby("level")
    .agg(
        competitive_groups=("field_size", "size"),
        median_field_size=("field_size", "median"),
        mean_field_size=("field_size", "mean"),
        max_field_size=("field_size", "max"),
    )
    .reindex(LEVEL_ORDER)
    .to_string()
)

COMPETITIVE GROUPS

Total groups: 10,686
Single-team groups: 4,085
Groups with 2 teams: 2,543
Groups with 3+ teams: 4,058

GROUPS WITH 3+ TEAMS BY LEVEL
       competitive_groups  median_field_size  mean_field_size  max_field_size
level                                                                        
1                    1049             4.0000           5.2459              27
2                    1088             4.0000           5.2803              30
3                     830             4.0000           5.7337              32
4                     486             4.0000           5.6070              29
4.2                   175             4.0000           5.4171              23
5                     250             4.0000           5.5120              24
6                     180             4.0000           5.1222              18


In [10]:
placement_df = analysis_df[
    analysis_df["field_size"] >= 3
].copy()

zero_deduction_placement_df = placement_df[
    placement_df["zero_deduction"]
].copy()

print(
    f"Placement-eligible performances: "
    f"{len(placement_df):,} "
    f"({len(placement_df) / len(analysis_df) * 100:.1f}%)"
)

print(
    f"Zero-deduction placement performances: "
    f"{len(zero_deduction_placement_df):,}"
)

print(
    f"Competitive groups: "
    f"{len(placement_df[group_cols].drop_duplicates()):,}"
)

print(
    f"Competitions represented: "
    f"{placement_df['competition_id'].nunique():,}"
)

print(
    f"Teams represented: "
    f"{placement_df['team_id'].nunique():,}"
)

print()
print("PLACEMENT-ELIGIBLE PERFORMANCES BY LEVEL")

placement_level_summary = (
    placement_df
    .groupby("level")
    .agg(
        performances=("team_id", "size"),
        zero_deduction_performances=(
            "zero_deduction",
            "sum",
        ),
        competitive_groups=(
            "division_id",
            "size",
        ),
    )
    .reindex(LEVEL_ORDER)
)

placement_level_summary["pct_zero_deduction"] = (
    placement_level_summary[
        "zero_deduction_performances"
    ]
    / placement_level_summary["performances"]
    * 100
)

print(
    placement_level_summary[
        [
            "performances",
            "zero_deduction_performances",
            "pct_zero_deduction",
        ]
    ].to_string()
)

Placement-eligible performances: 21,980 (70.6%)
Zero-deduction placement performances: 12,127
Competitive groups: 4,058
Competitions represented: 148
Teams represented: 4,112

PLACEMENT-ELIGIBLE PERFORMANCES BY LEVEL
       performances  zero_deduction_performances  pct_zero_deduction
level                                                               
1              5503                         3513             63.8379
2              5745                         3453             60.1044
3              4759                         2541             53.3936
4              2725                         1319             48.4037
4.2             948                          432             45.5696
5              1378                          601             43.6139
6               922                          268             29.0672


In [11]:
analysis_categories = [
    "pyramid_difficulty",
    "dance_difficulty",
    "stunt_execution",
    "pyramid_execution",
    "toss_execution",
    "standing_tumbling_execution",
    "running_tumbling_execution",
    "jump_execution",
    "rc",
    "formations_transitions",
    "dance_execution",
    "show",
]

winner_comparison = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_placement_df[
        zero_deduction_placement_df["level"] == level
    ]

    for col in analysis_categories:
        scores = level_df[
            ["is_winner", col]
        ].dropna()

        # Handles structurally non-applicable categories,
        # such as toss for Level 1.
        if scores.empty:
            continue

        winner_scores = scores.loc[
            scores["is_winner"],
            col,
        ]

        non_winner_scores = scores.loc[
            ~scores["is_winner"],
            col,
        ]

        if winner_scores.empty or non_winner_scores.empty:
            continue

        winner_comparison.append({
            "level": level,
            "category": col,
            "winner_n": len(winner_scores),
            "non_winner_n": len(non_winner_scores),
            "winner_mean": winner_scores.mean(),
            "non_winner_mean": non_winner_scores.mean(),
            "raw_difference": (
                winner_scores.mean()
                - non_winner_scores.mean()
            ),
        })

winner_comparison = pd.DataFrame(
    winner_comparison
)

print("WINNER VS NON-WINNER MEANS BY LEVEL")
print("(Zero-deduction performances; fields with 3+ teams)")

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        winner_comparison[
            winner_comparison["level"] == level
        ]
        .sort_values(
            "raw_difference",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "winner_n",
                "non_winner_n",
                "winner_mean",
                "non_winner_mean",
                "raw_difference",
            ],
        )
    )

WINNER VS NON-WINNER MEANS BY LEVEL
(Zero-deduction performances; fields with 3+ teams)

LEVEL 1
                   category  winner_n  non_winner_n  winner_mean  non_winner_mean  raw_difference
            stunt_execution       879          2634       3.8449           3.7646          0.0804
         pyramid_difficulty       879          2634       3.8344           3.7550          0.0793
 running_tumbling_execution       879          2634       3.8909           3.8120          0.0789
standing_tumbling_execution       879          2634       3.9139           3.8460          0.0679
                         rc       879          2634       1.8430           1.7759          0.0670
          pyramid_execution       879          2634       3.8501           3.7844          0.0657
                       show       879          2634       1.8266           1.7648          0.0618
            dance_execution       879          2634       0.8476           0.7929          0.0546
           dance_diff

In [12]:
standardized_comparison = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_placement_df[
        zero_deduction_placement_df["level"] == level
    ]

    for col in analysis_categories:
        scores = level_df[
            ["is_winner", col]
        ].dropna()

        if scores.empty:
            continue

        winner_scores = scores.loc[
            scores["is_winner"],
            col,
        ]

        non_winner_scores = scores.loc[
            ~scores["is_winner"],
            col,
        ]

        if (
            len(winner_scores) < 2
            or len(non_winner_scores) < 2
        ):
            continue

        pooled_std = np.sqrt(
            (
                winner_scores.var(ddof=1)
                + non_winner_scores.var(ddof=1)
            ) / 2
        )

        if pooled_std == 0:
            standardized_difference = np.nan
        else:
            standardized_difference = (
                winner_scores.mean()
                - non_winner_scores.mean()
            ) / pooled_std

        standardized_comparison.append({
            "level": level,
            "category": col,
            "winner_n": len(winner_scores),
            "non_winner_n": len(non_winner_scores),
            "winner_mean": winner_scores.mean(),
            "non_winner_mean": non_winner_scores.mean(),
            "raw_difference": (
                winner_scores.mean()
                - non_winner_scores.mean()
            ),
            "standardized_difference":
                standardized_difference,
        })

standardized_comparison = pd.DataFrame(
    standardized_comparison
)

print("STANDARDIZED WINNER DIFFERENCES BY LEVEL")
print("(Zero-deduction performances; fields with 3+ teams)")
print("(Positive = higher scores among winners)")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        standardized_comparison[
            standardized_comparison["level"] == level
        ]
        .sort_values(
            "standardized_difference",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "standardized_difference",
                "raw_difference",
                "winner_n",
                "non_winner_n",
            ],
        )
    )

STANDARDIZED WINNER DIFFERENCES BY LEVEL
(Zero-deduction performances; fields with 3+ teams)
(Positive = higher scores among winners)


LEVEL 1
                   category  standardized_difference  raw_difference  winner_n  non_winner_n
                         rc                   0.9543          0.0670       879          2634
                       show                   0.8547          0.0618       879          2634
 running_tumbling_execution                   0.7337          0.0789       879          2634
         pyramid_difficulty                   0.7040          0.0793       879          2634
            stunt_execution                   0.6741          0.0804       879          2634
standing_tumbling_execution                   0.6714          0.0679       879          2634
          pyramid_execution                   0.5693          0.0657       879          2634
            dance_execution                   0.5606          0.0546       879          2634
           dance_di

In [13]:
correlation_results = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_placement_df[
        zero_deduction_placement_df["level"] == level
    ]

    for col in analysis_categories:
        scores = level_df[
            [col, "placement_pct"]
        ].dropna()

        if len(scores) < 3:
            continue

        correlation = scores[
            col
        ].corr(
            scores["placement_pct"],
            method="spearman",
        )

        correlation_results.append({
            "level": level,
            "category": col,
            "performances": len(scores),
            "spearman_r": correlation,
        })

correlation_results = pd.DataFrame(
    correlation_results
)

print("SPEARMAN CORRELATIONS WITH PLACEMENT BY LEVEL")
print("(Zero-deduction performances; fields with 3+ teams)")
print("(Negative = higher category score associated with better placement)")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        correlation_results[
            correlation_results["level"] == level
        ]
        .sort_values(
            "spearman_r",
            ascending=True,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "performances",
                "spearman_r",
            ],
        )
    )

SPEARMAN CORRELATIONS WITH PLACEMENT BY LEVEL
(Zero-deduction performances; fields with 3+ teams)
(Negative = higher category score associated with better placement)


LEVEL 1
                   category  performances  spearman_r
                         rc          3513     -0.5116
                       show          3513     -0.4847
         pyramid_difficulty          3513     -0.4206
 running_tumbling_execution          3513     -0.4050
standing_tumbling_execution          3513     -0.4042
            stunt_execution          3513     -0.3714
            dance_execution          3513     -0.3498
           dance_difficulty          3513     -0.3103
             jump_execution          3513     -0.3019
          pyramid_execution          3513     -0.2910
     formations_transitions          3513     -0.1794

LEVEL 2
                   category  performances  spearman_r
                         rc          3453     -0.4951
                       show          3453     -0.4627
     

In [14]:
category_correlation_results = []

for level in LEVEL_ORDER:
    level_df = zero_deduction_placement_df[
        zero_deduction_placement_df["level"] == level
    ]

    corr_matrix = (
        level_df[analysis_categories]
        .corr(method="spearman")
    )

    for i, col_a in enumerate(analysis_categories):
        for col_b in analysis_categories[i + 1:]:
            correlation = corr_matrix.loc[
                col_a,
                col_b,
            ]

            if pd.isna(correlation):
                continue

            category_correlation_results.append({
                "level": level,
                "category_a": col_a,
                "category_b": col_b,
                "spearman_r": correlation,
                "abs_spearman_r": abs(correlation),
            })

category_correlation_results = pd.DataFrame(
    category_correlation_results
)

print("STRONGEST CATEGORY-PAIR CORRELATIONS BY LEVEL")
print("(Zero-deduction performances; fields with 3+ teams)")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        category_correlation_results[
            category_correlation_results["level"] == level
        ]
        .sort_values(
            "abs_spearman_r",
            ascending=False,
        )
        .head(10)
        .to_string(
            index=False,
            columns=[
                "category_a",
                "category_b",
                "spearman_r",
            ],
        )
    )

STRONGEST CATEGORY-PAIR CORRELATIONS BY LEVEL
(Zero-deduction performances; fields with 3+ teams)


LEVEL 1
                 category_a                 category_b  spearman_r
                         rc                       show      0.7308
standing_tumbling_execution running_tumbling_execution      0.5869
            stunt_execution          pyramid_execution      0.5687
            dance_execution                       show      0.5429
         pyramid_difficulty                         rc      0.5292
         pyramid_difficulty                       show      0.4601
                         rc            dance_execution      0.4533
 running_tumbling_execution             jump_execution      0.4317
standing_tumbling_execution             jump_execution      0.4289
            stunt_execution                         rc      0.4174

LEVEL 2
                 category_a                 category_b  spearman_r
                         rc                       show      0.7193
standing_tum

## Multivariable Analysis

Univariate relationships show which scoring categories individually distinguish
stronger competitive performances, but scoring categories are correlated with
one another. For example, routine composition and showmanship measures may
capture overlapping aspects of overall routine quality.

The multivariable analysis therefore evaluates which categories retain an
independent relationship with placement after accounting for the other scoring
components.

Models are fit separately by competitive level because identical numeric
category scores do not represent identical athletic difficulty across levels.

The primary analysis uses zero-deduction performances from fields containing
at least three teams. This isolates category-score relationships from recorded
deduction effects while preserving placement within the original competitive
context.

Because placement is normalized within competition, division, and round,
`placement_pct` is used as the outcome:

- `0.0` = first place
- `1.0` = last place

Predictors are standardized within competitive group before modeling so that
coefficients represent relative scoring advantages within the specific field
being judged rather than differences in scoring environments across events.

The resulting coefficients are interpreted as conditional associations, not
causal effects.

In [15]:
model_df = zero_deduction_placement_df.copy()

model_categories = analysis_categories.copy()

STD_EPSILON = 1e-8

# Standardize category scores within the actual competitive field.
# If a category has zero or effectively zero variation within a field,
# its standardized value is left undefined here and later assigned 0
# for modeling because it did not differentiate teams in that field.
for col in model_categories:
    group_mean = model_df.groupby(
        group_cols
    )[col].transform("mean")

    group_std = model_df.groupby(
        group_cols
    )[col].transform("std")

    valid_std = group_std.where(
        group_std > STD_EPSILON
    )

    model_df[f"{col}_z"] = (
        model_df[col] - group_mean
    ) / valid_std

model_feature_cols = [
    f"{col}_z"
    for col in model_categories
]

print("WITHIN-FIELD STANDARDIZATION")
print()

variation_summary = []

for level in LEVEL_ORDER:
    level_df = model_df[
        model_df["level"] == level
    ]

    for col in model_categories:
        z_col = f"{col}_z"

        applicable = level_df[col].notna()
        usable = (
            level_df[z_col]
            .replace([np.inf, -np.inf], np.nan)
            .notna()
        )

        variation_summary.append({
            "level": level,
            "category": col,
            "applicable_performances": applicable.sum(),
            "usable_standardized_values": usable.sum(),
            "pct_within_field_variation": (
                usable.sum()
                / applicable.sum()
                * 100
                if applicable.sum() > 0
                else np.nan
            ),
        })

variation_summary = pd.DataFrame(
    variation_summary
)

print(
    "Infinite standardized values:",
    np.isinf(
        model_df[model_feature_cols]
        .to_numpy()
    ).sum(),
)

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        variation_summary[
            variation_summary["level"] == level
        ]
        .sort_values(
            "pct_within_field_variation",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "applicable_performances",
                "usable_standardized_values",
                "pct_within_field_variation",
            ],
        )
    )

WITHIN-FIELD STANDARDIZATION

Infinite standardized values: 0

LEVEL 1
                   category  applicable_performances  usable_standardized_values  pct_within_field_variation
                         rc                     3513                        3265                     92.9405
                       show                     3513                        3232                     92.0011
         pyramid_difficulty                     3513                        3118                     88.7560
            stunt_execution                     3513                        3077                     87.5890
          pyramid_execution                     3513                        2946                     83.8599
 running_tumbling_execution                     3513                        2943                     83.7746
standing_tumbling_execution                     3513                        2934                     83.5184
            dance_execution                     3513     

In [16]:
level_model_features = {}
level_model_samples = {}

for level in LEVEL_ORDER:
    level_df = model_df[
        model_df["level"] == level
    ].copy()

    # Toss is structurally absent for Level 1.
    # For Level 2, retain toss only for toss-applicable divisions
    # rather than imputing a score for Mini.
    if level == "1":
        retained_categories = [
            col
            for col in model_categories
            if col != "toss_execution"
        ]
    elif level == "2":
        retained_categories = [
            col
            for col in model_categories
            if col != "toss_execution"
        ]
    else:
        retained_categories = (
            model_categories.copy()
        )

    retained_features = [
        f"{col}_z"
        for col in retained_categories
    ]

    # When a category is constant within a competitive field,
    # its standardized value is undefined. For modeling purposes,
    # assign 0: the performance has no relative advantage or
    # disadvantage in a category that did not differentiate
    # teams within that field.
    level_df[retained_features] = (
        level_df[retained_features]
        .fillna(0.0)
    )

    level_model_features[level] = (
        retained_categories
    )

    level_model_samples[level] = (
        level_df
    )

    print(f"\nLEVEL {level}")
    print(
        "Predictors:",
        ", ".join(retained_categories),
    )
    print(
        f"Model performances: "
        f"{len(level_df):,}"
    )
    print(
        f"Competitive groups: "
        f"{len(level_df[group_cols].drop_duplicates()):,}"
    )


LEVEL 1
Predictors: pyramid_difficulty, dance_difficulty, stunt_execution, pyramid_execution, standing_tumbling_execution, running_tumbling_execution, jump_execution, rc, formations_transitions, dance_execution, show
Model performances: 3,513
Competitive groups: 1,028

LEVEL 2
Predictors: pyramid_difficulty, dance_difficulty, stunt_execution, pyramid_execution, standing_tumbling_execution, running_tumbling_execution, jump_execution, rc, formations_transitions, dance_execution, show
Model performances: 3,453
Competitive groups: 1,024

LEVEL 3
Predictors: pyramid_difficulty, dance_difficulty, stunt_execution, pyramid_execution, toss_execution, standing_tumbling_execution, running_tumbling_execution, jump_execution, rc, formations_transitions, dance_execution, show
Model performances: 2,541
Competitive groups: 770

LEVEL 4
Predictors: pyramid_difficulty, dance_difficulty, stunt_execution, pyramid_execution, toss_execution, standing_tumbling_execution, running_tumbling_execution, jump_exe

In [17]:
from sklearn.linear_model import RidgeCV

RIDGE_ALPHAS = np.logspace(-3, 3, 100)

ridge_models = {}
ridge_results = []

for level in LEVEL_ORDER:
    level_df = level_model_samples[level]
    categories = level_model_features[level]

    feature_cols = [
        f"{col}_z"
        for col in categories
    ]

    X = level_df[feature_cols]
    y = level_df["placement_pct"]

    model = RidgeCV(
        alphas=RIDGE_ALPHAS,
    )

    model.fit(X, y)

    ridge_models[level] = model

    for category, coefficient in zip(
        categories,
        model.coef_,
    ):
        ridge_results.append({
            "level": level,
            "category": category,
            "coefficient": coefficient,
            "abs_coefficient": abs(coefficient),
        })

ridge_results = pd.DataFrame(
    ridge_results
)

print("RIDGE MULTIVARIABLE COEFFICIENTS")
print(
    "Negative coefficients indicate association "
    "with better placement."
)
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")
    print(
        f"Selected alpha: "
        f"{ridge_models[level].alpha_:.4f}"
    )

    print(
        ridge_results[
            ridge_results["level"] == level
        ]
        .sort_values(
            "abs_coefficient",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "coefficient",
            ],
        )
    )

RIDGE MULTIVARIABLE COEFFICIENTS
Negative coefficients indicate association with better placement.


LEVEL 1
Selected alpha: 70.5480
                   category  coefficient
         pyramid_difficulty      -0.0610
            stunt_execution      -0.0602
                         rc      -0.0596
                       show      -0.0516
standing_tumbling_execution      -0.0511
 running_tumbling_execution      -0.0502
            dance_execution      -0.0451
             jump_execution      -0.0436
          pyramid_execution      -0.0421
           dance_difficulty      -0.0403
     formations_transitions      -0.0275

LEVEL 2
Selected alpha: 70.5480
                   category  coefficient
                         rc      -0.0593
            stunt_execution      -0.0588
         pyramid_difficulty      -0.0552
standing_tumbling_execution      -0.0520
          pyramid_execution      -0.0498
 running_tumbling_execution      -0.0494
            dance_execution      -0.0444
              

### Competition-Held-Out Model Validation

The initial Ridge models describe conditional relationships within the observed
season, but model quality should be evaluated on competitions that were not used
to fit the model.

Randomly splitting individual performances could place performances from the
same competition in both training and validation data, allowing competition-
specific judging patterns to influence both sets.

Validation therefore uses grouped cross-validation in which entire competitions
are held out together. This provides a more realistic estimate of whether the
scoring relationships identified within each level generalize to other
competitions from the same season.

Model performance is evaluated using:

- **R²** — proportion of variation in normalized placement explained by the model.
- **MAE** — average absolute error in normalized placement.

Because `placement_pct` ranges from 0 to 1, MAE can be interpreted directly on
the normalized placement scale.

In [18]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
)

N_SPLITS = 5

cv_results = []
cv_fold_results = []

for level in LEVEL_ORDER:
    level_df = level_model_samples[level]
    categories = level_model_features[level]

    feature_cols = [
        f"{col}_z"
        for col in categories
    ]

    X = level_df[feature_cols]
    y = level_df["placement_pct"]
    groups = level_df["competition_id"]

    outer_cv = GroupKFold(
        n_splits=N_SPLITS
    )

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_groups = groups.iloc[train_idx]

        # Select alpha using only the training competitions.
        inner_cv = GroupKFold(
            n_splits=N_SPLITS
        )

        inner_splits = list(
            inner_cv.split(
                X_train,
                y_train,
                groups=train_groups,
            )
        )

        model = RidgeCV(
            alphas=RIDGE_ALPHAS,
            cv=inner_splits,
        )

        model.fit(
            X_train,
            y_train,
        )

        predictions = model.predict(
            X_test
        )

        cv_fold_results.append({
            "level": level,
            "fold": fold,
            "alpha": model.alpha_,
            "r2": r2_score(
                y_test,
                predictions,
            ),
            "mae": mean_absolute_error(
                y_test,
                predictions,
            ),
        })

cv_fold_results = pd.DataFrame(
    cv_fold_results
)

cv_results = (
    cv_fold_results
    .groupby("level", as_index=False)
    .agg(
        mean_r2=("r2", "mean"),
        std_r2=("r2", "std"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        median_alpha=("alpha", "median"),
    )
)

sample_summary = pd.DataFrame([
    {
        "level": level,
        "performances": len(
            level_model_samples[level]
        ),
        "competitions": (
            level_model_samples[level][
                "competition_id"
            ].nunique()
        ),
    }
    for level in LEVEL_ORDER
])

cv_results = sample_summary.merge(
    cv_results,
    on="level",
    how="left",
)

cv_results["level"] = pd.Categorical(
    cv_results["level"],
    categories=LEVEL_ORDER,
    ordered=True,
)

cv_results = cv_results.sort_values(
    "level"
)

print(
    "NESTED COMPETITION-HELD-OUT "
    "RIDGE VALIDATION"
)
print()

print(
    cv_results.to_string(
        index=False,
        columns=[
            "level",
            "performances",
            "competitions",
            "mean_r2",
            "std_r2",
            "mean_mae",
            "std_mae",
            "median_alpha",
        ],
    )
)

NESTED COMPETITION-HELD-OUT RIDGE VALIDATION

level  performances  competitions  mean_r2  std_r2  mean_mae  std_mae  median_alpha
    1          3513           129   0.6360  0.0169    0.1591   0.0062       93.2603
    2          3453           121   0.6039  0.0155    0.1618   0.0059       93.2603
    3          2541           101   0.6009  0.0318    0.1535   0.0103      107.2267
    4          1319            71   0.5722  0.0384    0.1500   0.0087       81.1131
  4.2           432            40   0.5800  0.1045    0.1465   0.0166       40.3702
    5           601            36   0.5048  0.1133    0.1604   0.0245       46.4159
    6           268            24   0.2998  0.2547    0.2012   0.0412       35.1119


### Coefficient Stability Across Competitions

Predictive performance alone does not establish that individual scoring
categories have stable relationships with placement.

To evaluate the robustness of the multivariable findings, Ridge coefficients
are examined across the training models created during competition-held-out
cross-validation.

A category whose coefficient remains similar in direction and magnitude across
folds provides stronger evidence of a season-wide relationship with placement.
Categories whose coefficients vary substantially across folds are interpreted
more cautiously, particularly at levels with fewer competitions.

Because lower `placement_pct` represents better placement, negative
coefficients indicate that higher relative category scores are associated with
better competitive outcomes.

In [19]:
coefficient_stability = []

for level in LEVEL_ORDER:
    level_df = level_model_samples[level]
    categories = level_model_features[level]

    feature_cols = [
        f"{col}_z"
        for col in categories
    ]

    X = level_df[feature_cols]
    y = level_df["placement_pct"]
    groups = level_df["competition_id"]

    outer_cv = GroupKFold(
        n_splits=N_SPLITS
    )

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        train_groups = groups.iloc[train_idx]

        inner_cv = GroupKFold(
            n_splits=N_SPLITS
        )

        inner_splits = list(
            inner_cv.split(
                X_train,
                y_train,
                groups=train_groups,
            )
        )

        model = RidgeCV(
            alphas=RIDGE_ALPHAS,
            cv=inner_splits,
        )

        model.fit(
            X_train,
            y_train,
        )

        for category, coefficient in zip(
            categories,
            model.coef_,
        ):
            coefficient_stability.append({
                "level": level,
                "fold": fold,
                "category": category,
                "coefficient": coefficient,
            })

coefficient_stability = pd.DataFrame(
    coefficient_stability
)

stability_summary = (
    coefficient_stability
    .groupby(
        ["level", "category"],
        as_index=False,
    )
    .agg(
        mean_coefficient=(
            "coefficient",
            "mean",
        ),
        std_coefficient=(
            "coefficient",
            "std",
        ),
        min_coefficient=(
            "coefficient",
            "min",
        ),
        max_coefficient=(
            "coefficient",
            "max",
        ),
        negative_folds=(
            "coefficient",
            lambda x: (x < 0).sum(),
        ),
    )
)

stability_summary["abs_mean_coefficient"] = (
    stability_summary[
        "mean_coefficient"
    ].abs()
)

print("RIDGE COEFFICIENT STABILITY")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        stability_summary[
            stability_summary["level"] == level
        ]
        .sort_values(
            "abs_mean_coefficient",
            ascending=False,
        )
        .to_string(
            index=False,
            columns=[
                "category",
                "mean_coefficient",
                "std_coefficient",
                "min_coefficient",
                "max_coefficient",
                "negative_folds",
            ],
        )
    )

RIDGE COEFFICIENT STABILITY


LEVEL 1
                   category  mean_coefficient  std_coefficient  min_coefficient  max_coefficient  negative_folds
         pyramid_difficulty           -0.0603           0.0020          -0.0628          -0.0578               5
            stunt_execution           -0.0596           0.0014          -0.0609          -0.0574               5
                         rc           -0.0595           0.0021          -0.0619          -0.0563               5
                       show           -0.0518           0.0009          -0.0532          -0.0508               5
standing_tumbling_execution           -0.0508           0.0034          -0.0548          -0.0461               5
 running_tumbling_execution           -0.0500           0.0027          -0.0528          -0.0465               5
            dance_execution           -0.0449           0.0043          -0.0494          -0.0382               5
             jump_execution           -0.0433           0.

### Multivariable Findings

Competition-held-out validation indicates that scoring-component relationships
generalize across competitions for Levels 1–5 and Level 4.2, although model
stability decreases as sample size declines at the upper levels. Level 6
produces substantially weaker and more variable out-of-sample performance and
is therefore interpreted more cautiously.

Across five competition-held-out folds, every included scoring category retained
a negative coefficient at every level. Higher relative category scores were
therefore consistently associated with better relative placement after
accounting for the other scoring components.

Several broader patterns emerge:

- **Stunt execution is consistently important across levels.** It remains among
  the stronger conditional relationships at every level and is particularly
  prominent in Level 4.2.

- **Pyramid difficulty is a persistent differentiator through Level 5.** Its
  relationship remains strong after controlling for correlated execution and
  performance categories, although its conditional relationship is much weaker
  in Level 6.

- **Routine Composition (RC) and Show remain meaningful after controlling for
  their strong correlation.** Their univariate relationships therefore cannot
  be treated as independent, but neither relationship disappears entirely in
  the multivariable models.

- **Running tumbling execution becomes increasingly prominent in some higher
  levels.** It is among the strongest conditional relationships in Level 3 and
  is the strongest Level 6 coefficient across all validation folds.

- **Formations and transitions generally show one of the weaker relationships
  with placement through Level 5.** Its larger Level 6 coefficient should be
  interpreted cautiously because Level 6 has the smallest sample and weakest
  out-of-sample model performance.

These coefficients describe conditional associations within the scoring system,
not causal effects. Placement is derived from the same scored routine components,
so the analysis identifies which components most consistently differentiate
competitive outcomes rather than estimating what would happen if a team
independently increased one category while holding all other routine qualities
constant.

## Deduction Impact on Competitive Outcomes

The zero-deduction analysis isolates scoring-component relationships from
recorded penalties, but deductions are themselves an important component of
competitive outcomes.

This section returns to all placement-eligible performances to examine how
recorded deductions relate to relative placement and whether that relationship
differs across competitive levels.

Because deductions can have a dual effect—a direct penalty and the possible
loss of difficulty credit when an intended skill is not successfully
completed—the analysis does not interpret the deduction value as the total
effect of an error.

Instead, it evaluates the observed competitive outcomes associated with
performances receiving different deduction amounts.

In [20]:
deduction_impact_summary = (
    placement_df
    .groupby("level")
    .agg(
        performances=(
            "team_id",
            "size",
        ),
        zero_deduction_rate=(
            "zero_deduction",
            "mean",
        ),
        mean_deductions=(
            "deductions",
            "mean",
        ),
        median_deductions=(
            "deductions",
            "median",
        ),
        mean_placement_pct=(
            "placement_pct",
            "mean",
        ),
    )
    .reindex(LEVEL_ORDER)
)

deduction_impact_summary[
    "zero_deduction_rate"
] *= 100

print("DEDUCTION CONTEXT BY LEVEL")
print()

print(
    deduction_impact_summary.to_string()
)

print()
print(
    "SPEARMAN CORRELATION: "
    "DEDUCTIONS VS PLACEMENT"
)
print(
    "(Positive = more deductions associated "
    "with worse placement)"
)
print()

deduction_correlations = []

for level in LEVEL_ORDER:
    level_df = placement_df[
        placement_df["level"] == level
    ]

    correlation = (
        level_df["deductions"]
        .corr(
            level_df["placement_pct"],
            method="spearman",
        )
    )

    deduction_correlations.append({
        "level": level,
        "performances": len(level_df),
        "spearman_correlation": correlation,
    })

deduction_correlations = pd.DataFrame(
    deduction_correlations
)

print(
    deduction_correlations.to_string(
        index=False
    )
)

DEDUCTION CONTEXT BY LEVEL

       performances  zero_deduction_rate  mean_deductions  median_deductions  mean_placement_pct
level                                                                                           
1              5503              63.8379           0.2338             0.0000              0.4974
2              5745              60.1044           0.3295             0.0000              0.4983
3              4759              53.3936           0.3978             0.0000              0.4981
4              2725              48.4037           0.4953             0.1500              0.4986
4.2             948              45.5696           0.5686             0.2500              0.4986
5              1378              43.6139           0.5542             0.2500              0.4990
6               922              29.0672           0.7224             0.4000              0.4987

SPEARMAN CORRELATION: DEDUCTIONS VS PLACEMENT
(Positive = more deductions associated with worse pl

In [21]:
deduction_group_comparison = []

for level in LEVEL_ORDER:
    level_df = placement_df[
        placement_df["level"] == level
    ]

    zero_df = level_df[
        level_df["deductions"] == 0
    ]

    deduction_df = level_df[
        level_df["deductions"] > 0
    ]

    deduction_group_comparison.append({
        "level": level,
        "zero_deduction_n": len(zero_df),
        "deduction_n": len(deduction_df),
        "zero_deduction_mean_placement": (
            zero_df["placement_pct"].mean()
        ),
        "deduction_mean_placement": (
            deduction_df["placement_pct"].mean()
        ),
        "placement_difference": (
            deduction_df["placement_pct"].mean()
            - zero_df["placement_pct"].mean()
        ),
        "zero_deduction_win_rate": (
            zero_df["is_winner"].mean() * 100
        ),
        "deduction_win_rate": (
            deduction_df["is_winner"].mean() * 100
        ),
    })

deduction_group_comparison = pd.DataFrame(
    deduction_group_comparison
)

print(
    "ZERO-DEDUCTION VS DEDUCTION "
    "COMPETITIVE OUTCOMES"
)
print()

print(
    deduction_group_comparison.to_string(
        index=False
    )
)

ZERO-DEDUCTION VS DEDUCTION COMPETITIVE OUTCOMES

level  zero_deduction_n  deduction_n  zero_deduction_mean_placement  deduction_mean_placement  placement_difference  zero_deduction_win_rate  deduction_win_rate
    1              3513         1990                         0.4070                    0.6569                0.2499                  25.0213              9.2965
    2              3453         2292                         0.3889                    0.6633                0.2744                  26.0064              8.7260
    3              2541         2218                         0.3636                    0.6523                0.2887                  26.0527              8.1154
    4              1319         1406                         0.3241                    0.6623                0.3381                  28.8855              7.4680
  4.2               432          516                         0.3150                    0.6523                0.3373                  29.1667     

In [22]:
deduction_distribution = []

for level in LEVEL_ORDER:
    level_df = placement_df[
        placement_df["level"] == level
    ]

    deducted_df = level_df[
        level_df["deductions"] > 0
    ]

    deduction_distribution.append({
        "level": level,
        "deduction_performances": len(
            deducted_df
        ),
        "min": deducted_df[
            "deductions"
        ].min(),
        "q25": deducted_df[
            "deductions"
        ].quantile(0.25),
        "median": deducted_df[
            "deductions"
        ].median(),
        "q75": deducted_df[
            "deductions"
        ].quantile(0.75),
        "q90": deducted_df[
            "deductions"
        ].quantile(0.90),
        "q95": deducted_df[
            "deductions"
        ].quantile(0.95),
        "max": deducted_df[
            "deductions"
        ].max(),
        "unique_amounts": deducted_df[
            "deductions"
        ].nunique(),
    })

deduction_distribution = pd.DataFrame(
    deduction_distribution
)

print("POSITIVE DEDUCTION DISTRIBUTION BY LEVEL")
print()

print(
    deduction_distribution.to_string(
        index=False
    )
)

POSITIVE DEDUCTION DISTRIBUTION BY LEVEL

level  deduction_performances    min    q25  median    q75    q90    q95    max  unique_amounts
    1                    1990 0.0500 0.2500  0.3500 0.9000 1.5000 2.0000 5.9000              58
    2                    2292 0.0500 0.2500  0.7500 1.2500 1.7500 2.2500 7.5000              68
    3                    2218 0.0100 0.2500  0.7500 1.2500 1.7500 2.3500 6.0000              82
    4                    1406 0.0500 0.2500  0.7500 1.3500 2.0000 2.5000 5.7500              62
  4.2                     516 0.0500 0.5000  0.7500 1.5000 2.2500 2.7500 5.2500              36
    5                     777 0.0500 0.2500  0.7500 1.2500 2.0000 2.5900 8.2500              54
    6                     654 0.0500 0.2500  0.7500 1.5000 2.1500 2.9350 4.6000              57


In [23]:
deduction_bins = [
    -0.001,
    0,
    0.25,
    0.75,
    1.50,
    np.inf,
]

deduction_labels = [
    "0",
    "0.01–0.25",
    "0.26–0.75",
    "0.76–1.50",
    ">1.50",
]

placement_df["deduction_band"] = pd.cut(
    placement_df["deductions"],
    bins=deduction_bins,
    labels=deduction_labels,
    include_lowest=True,
)

deduction_band_summary = (
    placement_df
    .groupby(
        ["level", "deduction_band"],
        observed=True,
    )
    .agg(
        performances=("team_id", "size"),
        mean_deductions=("deductions", "mean"),
        mean_placement_pct=(
            "placement_pct",
            "mean",
        ),
        median_placement_pct=(
            "placement_pct",
            "median",
        ),
        win_rate=("is_winner", "mean"),
        top_3_rate=("top_3", "mean"),
    )
    .reset_index()
)

deduction_band_summary[
    ["win_rate", "top_3_rate"]
] *= 100

print("DEDUCTION DOSE-RESPONSE BY LEVEL")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        deduction_band_summary[
            deduction_band_summary["level"] == level
        ]
        .to_string(
            index=False,
            columns=[
                "deduction_band",
                "performances",
                "mean_deductions",
                "mean_placement_pct",
                "median_placement_pct",
                "win_rate",
                "top_3_rate",
            ],
        )
    )

DEDUCTION DOSE-RESPONSE BY LEVEL


LEVEL 1
deduction_band  performances  mean_deductions  mean_placement_pct  median_placement_pct  win_rate  top_3_rate
             0          3513           0.0000              0.4070                0.3750   25.0213     64.7310
     0.01–0.25           971           0.2058              0.5604                0.5714   13.8002     52.1112
     0.26–0.75           489           0.6252              0.6843                0.7500    6.7485     42.1268
     0.76–1.50           383           1.1860              0.7777                0.8636    4.1775     34.4648
         >1.50           147           2.2224              0.8883                1.0000    1.3605     29.9320

LEVEL 2
deduction_band  performances  mean_deductions  mean_placement_pct  median_placement_pct  win_rate  top_3_rate
             0          3453           0.0000              0.3889                0.3333   26.0064     65.1607
     0.01–0.25           805           0.2107              0.5200   

In [24]:
all_model_df = placement_df.copy()

STD_EPSILON = 1e-8

for col in model_categories:
    group_mean = all_model_df.groupby(
        group_cols
    )[col].transform("mean")

    group_std = all_model_df.groupby(
        group_cols
    )[col].transform("std")

    valid_std = group_std.where(
        group_std > STD_EPSILON
    )

    all_model_df[f"{col}_z"] = (
        all_model_df[col] - group_mean
    ) / valid_std

all_level_model_features = {}
all_level_model_samples = {}

for level in LEVEL_ORDER:
    level_df = all_model_df[
        all_model_df["level"] == level
    ].copy()

    if level in ["1", "2"]:
        retained_categories = [
            col
            for col in model_categories
            if col != "toss_execution"
        ]
    else:
        retained_categories = (
            model_categories.copy()
        )

    retained_features = [
        f"{col}_z"
        for col in retained_categories
    ]

    level_df[retained_features] = (
        level_df[retained_features]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0.0)
    )

    all_level_model_features[level] = (
        retained_categories
    )

    all_level_model_samples[level] = (
        level_df
    )

    print(f"\nLEVEL {level}")
    print(
        f"Model performances: "
        f"{len(level_df):,}"
    )
    print(
        f"Competitive groups: "
        f"{len(level_df[group_cols].drop_duplicates()):,}"
    )
    print(
        f"Competitions: "
        f"{level_df['competition_id'].nunique():,}"
    )
    print(
        "Infinite predictor values:",
        np.isinf(
            level_df[retained_features]
            .to_numpy()
        ).sum(),
    )



LEVEL 1
Model performances: 5,503
Competitive groups: 1,049
Competitions: 131
Infinite predictor values: 0

LEVEL 2
Model performances: 5,745
Competitive groups: 1,088
Competitions: 125
Infinite predictor values: 0

LEVEL 3
Model performances: 4,759
Competitive groups: 830
Competitions: 108
Infinite predictor values: 0

LEVEL 4
Model performances: 2,725
Competitive groups: 486
Competitions: 75
Infinite predictor values: 0

LEVEL 4.2
Model performances: 948
Competitive groups: 175
Competitions: 44
Infinite predictor values: 0

LEVEL 5
Model performances: 1,378
Competitive groups: 250
Competitions: 39
Infinite predictor values: 0

LEVEL 6
Model performances: 922
Competitive groups: 180
Competitions: 28
Infinite predictor values: 0


In [25]:
deduction_model_comparison = []

for level in LEVEL_ORDER:
    level_df = all_level_model_samples[level]
    categories = all_level_model_features[level]

    category_features = [
        f"{col}_z"
        for col in categories
    ]

    X_categories = level_df[
        category_features
    ].copy()

    X_with_deductions = (
        X_categories.copy()
    )

    X_with_deductions[
        "deductions"
    ] = level_df["deductions"]

    y = level_df["placement_pct"]
    groups = level_df["competition_id"]

    outer_cv = GroupKFold(
        n_splits=N_SPLITS
    )

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(
            X_categories,
            y,
            groups=groups,
        ),
        start=1,
    ):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_groups = groups.iloc[
            train_idx
        ]

        inner_cv = GroupKFold(
            n_splits=N_SPLITS
        )

        inner_splits = list(
            inner_cv.split(
                X_categories.iloc[
                    train_idx
                ],
                y_train,
                groups=train_groups,
            )
        )

        for model_name, X in [
            (
                "Categories only",
                X_categories,
            ),
            (
                "Categories + deductions",
                X_with_deductions,
            ),
        ]:
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            model = RidgeCV(
                alphas=RIDGE_ALPHAS,
                cv=inner_splits,
            )

            model.fit(
                X_train,
                y_train,
            )

            predictions = model.predict(
                X_test
            )

            deduction_model_comparison.append({
                "level": level,
                "fold": fold,
                "model": model_name,
                "r2": r2_score(
                    y_test,
                    predictions,
                ),
                "mae": mean_absolute_error(
                    y_test,
                    predictions,
                ),
            })

deduction_model_comparison = pd.DataFrame(
    deduction_model_comparison
)

comparison_summary = (
    deduction_model_comparison
    .groupby(
        ["level", "model"],
        as_index=False,
    )
    .agg(
        mean_r2=("r2", "mean"),
        std_r2=("r2", "std"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
    )
)

print(
    "INCREMENTAL VALUE OF DEDUCTIONS"
)
print()

for level in LEVEL_ORDER:
    level_results = (
        comparison_summary[
            comparison_summary[
                "level"
            ] == level
        ]
        .copy()
    )

    print(f"\nLEVEL {level}")

    print(
        level_results.to_string(
            index=False,
            columns=[
                "model",
                "mean_r2",
                "std_r2",
                "mean_mae",
                "std_mae",
            ],
        )
    )

    category_r2 = level_results.loc[
        level_results["model"]
        == "Categories only",
        "mean_r2",
    ].iloc[0]

    deduction_r2 = level_results.loc[
        level_results["model"]
        == "Categories + deductions",
        "mean_r2",
    ].iloc[0]

    category_mae = level_results.loc[
        level_results["model"]
        == "Categories only",
        "mean_mae",
    ].iloc[0]

    deduction_mae = level_results.loc[
        level_results["model"]
        == "Categories + deductions",
        "mean_mae",
    ].iloc[0]

    print(
        f"Δ R²: "
        f"{deduction_r2 - category_r2:+.4f}"
    )

    print(
        f"Δ MAE: "
        f"{deduction_mae - category_mae:+.4f}"
    )

INCREMENTAL VALUE OF DEDUCTIONS


LEVEL 1
                  model  mean_r2  std_r2  mean_mae  std_mae
Categories + deductions   0.7553  0.0257    0.1362   0.0112
        Categories only   0.7361  0.0284    0.1414   0.0119
Δ R²: +0.0192
Δ MAE: -0.0052

LEVEL 2
                  model  mean_r2  std_r2  mean_mae  std_mae
Categories + deductions   0.7272  0.0254    0.1426   0.0098
        Categories only   0.7042  0.0336    0.1493   0.0107
Δ R²: +0.0230
Δ MAE: -0.0066

LEVEL 3
                  model  mean_r2  std_r2  mean_mae  std_mae
Categories + deductions   0.7235  0.0193    0.1413   0.0078
        Categories only   0.7023  0.0176    0.1474   0.0070
Δ R²: +0.0212
Δ MAE: -0.0062

LEVEL 4
                  model  mean_r2  std_r2  mean_mae  std_mae
Categories + deductions   0.7474  0.0088    0.1361   0.0024
        Categories only   0.7138  0.0091    0.1462   0.0046
Δ R²: +0.0336
Δ MAE: -0.0101

LEVEL 4.2
                  model  mean_r2  std_r2  mean_mae  std_mae
Categories + deductions 

### Deduction Findings

Deductions show a consistent and substantial relationship with competitive
outcomes across all levels.

Zero-deduction performances have substantially better average normalized
placement and higher win rates than performances with recorded deductions.
The relationship also follows a clear dose-response pattern: at every level,
progressively larger deduction bands correspond to progressively worse mean
relative placement.

This relationship is not explained entirely by reduced category scores.
Competition-held-out model comparisons show that adding deduction amount to
the scored routine categories improves out-of-sample placement prediction at
every level.

The increase in mean held-out R² ranges from approximately 0.02 to 0.03 across
levels, while mean absolute error decreases in every case. The largest
incremental improvements occur around Levels 4, 4.2, and 5.

This supports the interpretation that competitive errors can influence outcomes
through two related mechanisms:

1. unsuccessful or uncredited skills may reduce the underlying category score;
2. recorded deductions impose an additional penalty beyond those category-score
   differences.

The analysis does not estimate the causal effect of an individual mistake,
because the occurrence and severity of errors are related to routine difficulty,
execution quality, and other characteristics of the performance. It does show
that deductions contain consistent competitive information beyond the scored
routine components themselves.

In [26]:
context_candidates = [
    "competition_name",
    "competition_id",
    "competition_date",
    "city",
    "state",
    "region",
    "event_type",
    "brand",
    "producer",
    "division_id",
    "division_name",
    "field_size",
]

print("AVAILABLE COMPETITIVE-CONTEXT COLUMNS")
print()

for col in context_candidates:
    if col in analysis_df.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col}")

print()
print("ALL DATASET COLUMNS")
print()

for col in analysis_df.columns:
    print(col)

AVAILABLE COMPETITIVE-CONTEXT COLUMNS

✗ competition_name
✓ competition_id
✗ competition_date
✗ city
✗ state
✗ region
✗ event_type
✗ brand
✗ producer
✓ division_id
✗ division_name
✓ field_size

ALL DATASET COLUMNS

competition_id
division_id
team_id
level
division_name_raw
age_group
size
is_coed
is_flex
is_d2
division_split
round
program_name
team_name
rank
raw_score
deductions
performance_score
event_score
toss_applicable
scorecard_max
scoring_schema
stunt_difficulty
stunt_execution
stunt_dod
stunt_max
pyramid_difficulty
pyramid_execution
toss_difficulty
toss_execution
standing_tumbling_difficulty
standing_tumbling_execution
standing_tumbling_dod
running_tumbling_difficulty
running_tumbling_execution
running_tumbling_dod
running_tumbling_max
jump_difficulty
jump_execution
rc
formations_transitions
dance_difficulty
dance_execution
show
zero_deduction
field_size
placement_pct
is_winner
top_3
top_quartile


In [27]:
field_size_distribution = (
    placement_df[
        group_cols + ["field_size"]
    ]
    .drop_duplicates()
    .groupby("level")["field_size"]
    .agg(
        competitive_groups="size",
        min_field="min",
        median_field="median",
        mean_field="mean",
        max_field="max",
    )
    .reindex(LEVEL_ORDER)
)

field_size_quantiles = (
    placement_df[
        group_cols + ["field_size"]
    ]
    .drop_duplicates()
    .groupby("level")["field_size"]
    .quantile(
        [0.25, 0.50, 0.75, 0.90]
    )
    .unstack()
    .reindex(LEVEL_ORDER)
)

field_size_quantiles.columns = [
    "q25",
    "q50",
    "q75",
    "q90",
]

field_size_summary = (
    field_size_distribution.join(
        field_size_quantiles
    )
)

print("COMPETITIVE FIELD SIZE BY LEVEL")
print()

print(
    field_size_summary.to_string()
)

print()
print("GROUP COUNTS BY EXACT FIELD SIZE")
print()

field_size_counts = (
    placement_df[
        group_cols + ["field_size"]
    ]
    .drop_duplicates()
    .groupby(
        ["level", "field_size"]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(LEVEL_ORDER)
)

print(
    field_size_counts.to_string()
)

COMPETITIVE FIELD SIZE BY LEVEL

       competitive_groups  min_field  median_field  mean_field  max_field    q25    q50    q75     q90
level                                                                                                 
1                    1049          3        4.0000      5.2459         27 3.0000 4.0000 6.0000  9.0000
2                    1088          3        4.0000      5.2803         30 3.0000 4.0000 6.0000  9.0000
3                     830          3        4.0000      5.7337         32 3.0000 4.0000 7.0000 10.0000
4                     486          3        4.0000      5.6070         29 3.0000 4.0000 7.0000 10.0000
4.2                   175          3        4.0000      5.4171         23 3.0000 4.0000 6.5000 10.0000
5                     250          3        4.0000      5.5120         24 3.0000 4.0000 6.7500 12.0000
6                     180          3        4.0000      5.1222         18 3.0000 4.0000 6.0000 10.0000

GROUP COUNTS BY EXACT FIELD SIZE

field

In [28]:
field_bins = [
    2,
    4,
    7,
    np.inf,
]

field_labels = [
    "Small (3–4)",
    "Medium (5–7)",
    "Large (8+)",
]

placement_df["field_size_band"] = pd.cut(
    placement_df["field_size"],
    bins=field_bins,
    labels=field_labels,
)

field_band_groups = (
    placement_df[
        group_cols
        + [
            "field_size",
            "field_size_band",
        ]
    ]
    .drop_duplicates()
    .groupby(
        ["level", "field_size_band"],
        observed=True,
    )
    .agg(
        competitive_groups=(
            "division_id",
            "size",
        ),
        performances=(
            "field_size",
            "sum",
        ),
        median_field_size=(
            "field_size",
            "median",
        ),
        max_field_size=(
            "field_size",
            "max",
        ),
    )
    .reset_index()
)

print("FIELD-SIZE BAND SAMPLE SIZES")
print()

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    print(
        field_band_groups[
            field_band_groups["level"] == level
        ]
        .to_string(
            index=False,
            columns=[
                "field_size_band",
                "competitive_groups",
                "performances",
                "median_field_size",
                "max_field_size",
            ],
        )
    )

FIELD-SIZE BAND SAMPLE SIZES


LEVEL 1
field_size_band  competitive_groups  performances  median_field_size  max_field_size
    Small (3–4)                 574          1936             3.0000               4
   Medium (5–7)                 289          1696             6.0000               7
     Large (8+)                 186          1871             9.0000              27

LEVEL 2
field_size_band  competitive_groups  performances  median_field_size  max_field_size
    Small (3–4)                 626          2129             3.0000               4
   Medium (5–7)                 263          1523             6.0000               7
     Large (8+)                 199          2093             9.0000              30

LEVEL 3
field_size_band  competitive_groups  performances  median_field_size  max_field_size
    Small (3–4)                 419          1433             3.0000               4
   Medium (5–7)                 255          1483             6.0000               7
     Lar

In [29]:
field_context_df = (
    zero_deduction_placement_df.copy()
)

field_context_df["field_size_band"] = pd.cut(
    field_context_df["field_size"],
    bins=field_bins,
    labels=field_labels,
)

field_context_correlations = []

for level in LEVEL_ORDER:
    for band in field_labels:
        band_df = field_context_df[
            (field_context_df["level"] == level)
            & (
                field_context_df[
                    "field_size_band"
                ] == band
            )
        ]

        for category in analysis_categories:
            category_df = band_df[
                [
                    category,
                    "placement_pct",
                ]
            ].dropna()

            if len(category_df) < 20:
                continue

            correlation = (
                category_df[category]
                .corr(
                    category_df[
                        "placement_pct"
                    ],
                    method="spearman",
                )
            )

            field_context_correlations.append({
                "level": level,
                "field_size_band": band,
                "category": category,
                "performances": len(
                    category_df
                ),
                "spearman_correlation": (
                    correlation
                ),
                "abs_correlation": abs(
                    correlation
                ),
            })

field_context_correlations = pd.DataFrame(
    field_context_correlations
)

print(
    "CATEGORY-PLACEMENT CORRELATIONS "
    "BY FIELD SIZE"
)
print(
    "Negative = higher score associated "
    "with better placement"
)

for level in LEVEL_ORDER:
    print(f"\n{'=' * 70}")
    print(f"LEVEL {level}")
    print("=" * 70)

    for band in field_labels:
        print(f"\n{band}")

        results = (
            field_context_correlations[
                (
                    field_context_correlations[
                        "level"
                    ] == level
                )
                & (
                    field_context_correlations[
                        "field_size_band"
                    ] == band
                )
            ]
            .sort_values(
                "abs_correlation",
                ascending=False,
            )
        )

        print(
            results.to_string(
                index=False,
                columns=[
                    "category",
                    "performances",
                    "spearman_correlation",
                ],
            )
        )

CATEGORY-PLACEMENT CORRELATIONS BY FIELD SIZE
Negative = higher score associated with better placement

LEVEL 1

Small (3–4)
                   category  performances  spearman_correlation
                         rc          1205               -0.4612
                       show          1205               -0.4014
         pyramid_difficulty          1205               -0.3786
standing_tumbling_execution          1205               -0.3394
 running_tumbling_execution          1205               -0.3363
            stunt_execution          1205               -0.3084
            dance_execution          1205               -0.2799
           dance_difficulty          1205               -0.2737
             jump_execution          1205               -0.2607
          pyramid_execution          1205               -0.2345
     formations_transitions          1205               -0.1476

Medium (5–7)
                   category  performances  spearman_correlation
                         rc  

### Scoring Differentiation by Field Size

Univariate relationships between category scores and placement generally become
stronger as competitive fields increase in size, particularly through Levels
1–4.

This pattern should not automatically be interpreted as evidence that judges
weight categories differently in larger fields. Normalized placement has much
coarser resolution in small fields, and larger fields may also differ in team
quality, event characteristics, or competitive depth.

Multivariable models are therefore used to examine whether the relative
importance of scoring categories changes across small, medium, and large
competitive fields after accounting for correlations among scoring components.

Field-size groups are defined consistently across levels:

- **Small:** 3–4 teams
- **Medium:** 5–7 teams
- **Large:** 8 or more teams

The analysis remains restricted to zero-deduction performances.

In [30]:
field_model_df = (
    zero_deduction_placement_df.copy()
)

field_model_df["field_size_band"] = pd.cut(
    field_model_df["field_size"],
    bins=field_bins,
    labels=field_labels,
)

field_model_samples = {}
field_model_features = {}

for level in LEVEL_ORDER:
    if level in ["1", "2"]:
        categories = [
            col
            for col in analysis_categories
            if col != "toss_execution"
        ]
    else:
        categories = (
            analysis_categories.copy()
        )

    for band in field_labels:
        sample = field_model_df[
            (field_model_df["level"] == level)
            & (
                field_model_df[
                    "field_size_band"
                ] == band
            )
        ].copy()

        for col in categories:
            group_mean = (
                sample.groupby(group_cols)[col]
                .transform("mean")
            )

            group_std = (
                sample.groupby(group_cols)[col]
                .transform("std")
            )

            valid_std = (
                group_std > STD_EPSILON
            )

            sample[f"{col}_z"] = np.where(
                valid_std,
                (
                    sample[col]
                    - group_mean
                )
                / group_std,
                np.nan,
            )

        predictor_cols = [
            f"{col}_z"
            for col in categories
        ]

        sample[predictor_cols] = (
            sample[predictor_cols]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .fillna(0)
        )

        field_model_samples[
            (level, band)
        ] = sample

        field_model_features[
            (level, band)
        ] = categories

print(
    "FIELD-SIZE MODEL SAMPLE CHECK"
)

for level in LEVEL_ORDER:
    print(f"\nLEVEL {level}")

    for band in field_labels:
        sample = field_model_samples[
            (level, band)
        ]

        print(
            f"{band:<14} | "
            f"{len(sample):>4,} performances | "
            f"{sample[group_cols].drop_duplicates().shape[0]:>3} groups | "
            f"{sample['competition_id'].nunique():>3} competitions"
        )

FIELD-SIZE MODEL SAMPLE CHECK

LEVEL 1
Small (3–4)    | 1,205 performances | 555 groups | 123 competitions
Medium (5–7)   | 1,073 performances | 287 groups |  62 competitions
Large (8+)     | 1,235 performances | 186 groups |  23 competitions

LEVEL 2
Small (3–4)    | 1,216 performances | 568 groups | 120 competitions
Medium (5–7)   |  916 performances | 257 groups |  50 competitions
Large (8+)     | 1,321 performances | 199 groups |  21 competitions

LEVEL 3
Small (3–4)    |  722 performances | 370 groups |  93 competitions
Medium (5–7)   |  783 performances | 244 groups |  52 competitions
Large (8+)     | 1,036 performances | 156 groups |  16 competitions

LEVEL 4
Small (3–4)    |  366 performances | 224 groups |  68 competitions
Medium (5–7)   |  377 performances | 133 groups |  31 competitions
Large (8+)     |  576 performances |  95 groups |   8 competitions

LEVEL 4.2
Small (3–4)    |  121 performances |  79 groups |  34 competitions
Medium (5–7)   |   83 performances |  31 group

In [31]:
field_interaction_df = (
    zero_deduction_placement_df.copy()
)

field_interaction_df[
    "log_field_size"
] = np.log1p(
    field_interaction_df["field_size"]
)

field_interaction_df[
    "log_field_size_centered"
] = (
    field_interaction_df[
        "log_field_size"
    ]
    - field_interaction_df.groupby(
        "level"
    )[
        "log_field_size"
    ].transform("mean")
)

print(
    "FIELD-SIZE INTERACTION VARIABLE"
)
print()

field_interaction_summary = (
    field_interaction_df
    .groupby("level")
    .agg(
        performances=(
            "team_id",
            "size",
        ),
        competitions=(
            "competition_id",
            "nunique",
        ),
        min_field_size=(
            "field_size",
            "min",
        ),
        median_field_size=(
            "field_size",
            "median",
        ),
        max_field_size=(
            "field_size",
            "max",
        ),
        mean_log_field_size=(
            "log_field_size",
            "mean",
        ),
    )
    .reindex(LEVEL_ORDER)
)

print(
    field_interaction_summary.to_string()
)

FIELD-SIZE INTERACTION VARIABLE

       performances  competitions  min_field_size  median_field_size  max_field_size  mean_log_field_size
level                                                                                                    
1              3513           129               3             6.0000              27               1.9632
2              3453           121               3             6.0000              30               1.9883
3              2541           101               3             7.0000              32               2.0915
4              1319            71               3             6.0000              29               2.0869
4.2             432            40               3             8.0000              23               2.1168
5               601            36               3             8.0000              24               2.1190
6               268            24               3             6.0000              18               1.9602


In [32]:
interaction_model_samples = {}
interaction_model_features = {}

for level in LEVEL_ORDER:
    level_df = field_interaction_df[
        field_interaction_df["level"] == level
    ].copy()

    if level in ["1", "2"]:
        categories = [
            col
            for col in analysis_categories
            if col != "toss_execution"
        ]
    else:
        categories = (
            analysis_categories.copy()
        )

    main_effect_features = []
    interaction_features = []

    for col in categories:
        group_mean = (
            level_df.groupby(group_cols)[col]
            .transform("mean")
        )

        group_std = (
            level_df.groupby(group_cols)[col]
            .transform("std")
        )

        valid_std = (
            group_std > STD_EPSILON
        )

        z_col = f"{col}_z"

        level_df[z_col] = np.where(
            valid_std,
            (
                level_df[col]
                - group_mean
            )
            / group_std,
            np.nan,
        )

        level_df[z_col] = (
            level_df[z_col]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .fillna(0)
        )

        interaction_col = (
            f"{col}_x_field_size"
        )

        level_df[interaction_col] = (
            level_df[z_col]
            * level_df[
                "log_field_size_centered"
            ]
        )

        main_effect_features.append(
            z_col
        )

        interaction_features.append(
            interaction_col
        )

    interaction_model_samples[
        level
    ] = level_df

    interaction_model_features[
        level
    ] = {
        "main": main_effect_features,
        "interactions": interaction_features,
    }

print(
    "FIELD-SIZE INTERACTION "
    "PREDICTORS BUILT"
)

for level in LEVEL_ORDER:
    sample = interaction_model_samples[
        level
    ]

    features = (
        interaction_model_features[
            level
        ]
    )

    predictor_cols = (
        features["main"]
        + ["log_field_size_centered"]
        + features["interactions"]
    )

    infinite_values = (
        np.isinf(
            sample[predictor_cols]
            .to_numpy()
        )
        .sum()
    )

    print(
        f"Level {level:<3} | "
        f"{len(sample):>4,} performances | "
        f"{sample['competition_id'].nunique():>3} competitions | "
        f"{len(features['interactions']):>2} interactions | "
        f"{infinite_values} infinite values"
    )

FIELD-SIZE INTERACTION PREDICTORS BUILT
Level 1   | 3,513 performances | 129 competitions | 11 interactions | 0 infinite values
Level 2   | 3,453 performances | 121 competitions | 11 interactions | 0 infinite values
Level 3   | 2,541 performances | 101 competitions | 12 interactions | 0 infinite values
Level 4   | 1,319 performances |  71 competitions | 12 interactions | 0 infinite values
Level 4.2 |  432 performances |  40 competitions | 12 interactions | 0 infinite values
Level 5   |  601 performances |  36 competitions | 12 interactions | 0 infinite values
Level 6   |  268 performances |  24 competitions | 12 interactions | 0 infinite values


In [33]:
field_interaction_comparison = []

for level in LEVEL_ORDER:
    level_df = interaction_model_samples[
        level
    ]

    features = interaction_model_features[
        level
    ]

    baseline_features = (
        features["main"]
        + ["log_field_size_centered"]
    )

    interaction_features = (
        baseline_features
        + features["interactions"]
    )

    X_baseline = level_df[
        baseline_features
    ]

    X_interaction = level_df[
        interaction_features
    ]

    y = level_df["placement_pct"]
    groups = level_df["competition_id"]

    outer_cv = GroupKFold(
        n_splits=N_SPLITS
    )

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        outer_cv.split(
            X_baseline,
            y,
            groups=groups,
        ),
        start=1,
    ):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_groups = groups.iloc[
            train_idx
        ]

        inner_cv = GroupKFold(
            n_splits=N_SPLITS
        )

        inner_splits = list(
            inner_cv.split(
                X_baseline.iloc[
                    train_idx
                ],
                y_train,
                groups=train_groups,
            )
        )

        for model_name, X in [
            (
                "Baseline",
                X_baseline,
            ),
            (
                "Field-size interactions",
                X_interaction,
            ),
        ]:
            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            model = RidgeCV(
                alphas=RIDGE_ALPHAS,
                cv=inner_splits,
            )

            model.fit(
                X_train,
                y_train,
            )

            predictions = model.predict(
                X_test
            )

            field_interaction_comparison.append({
                "level": level,
                "fold": fold,
                "model": model_name,
                "alpha": model.alpha_,
                "r2": r2_score(
                    y_test,
                    predictions,
                ),
                "mae": mean_absolute_error(
                    y_test,
                    predictions,
                ),
            })

field_interaction_comparison = pd.DataFrame(
    field_interaction_comparison
)

field_interaction_summary = (
    field_interaction_comparison
    .groupby(
        ["level", "model"],
        as_index=False,
    )
    .agg(
        mean_r2=("r2", "mean"),
        std_r2=("r2", "std"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        median_alpha=("alpha", "median"),
    )
)

print(
    "INCREMENTAL VALUE OF "
    "FIELD-SIZE INTERACTIONS"
)

for level in LEVEL_ORDER:
    level_results = (
        field_interaction_summary[
            field_interaction_summary[
                "level"
            ] == level
        ]
    )

    print(f"\nLEVEL {level}")

    print(
        level_results.to_string(
            index=False,
            columns=[
                "model",
                "mean_r2",
                "std_r2",
                "mean_mae",
                "std_mae",
                "median_alpha",
            ],
        )
    )

    baseline = level_results[
        level_results["model"]
        == "Baseline"
    ].iloc[0]

    interaction = level_results[
        level_results["model"]
        == "Field-size interactions"
    ].iloc[0]

    print(
        f"Δ R²: "
        f"{interaction['mean_r2'] - baseline['mean_r2']:+.4f}"
    )

    print(
        f"Δ MAE: "
        f"{interaction['mean_mae'] - baseline['mean_mae']:+.4f}"
    )

INCREMENTAL VALUE OF FIELD-SIZE INTERACTIONS

LEVEL 1
                  model  mean_r2  std_r2  mean_mae  std_mae  median_alpha
               Baseline   0.6361  0.0172    0.1589   0.0063       93.2603
Field-size interactions   0.6543  0.0167    0.1541   0.0062      123.2847
Δ R²: +0.0182
Δ MAE: -0.0048

LEVEL 2
                  model  mean_r2  std_r2  mean_mae  std_mae  median_alpha
               Baseline   0.6033  0.0150    0.1619   0.0055      123.2847
Field-size interactions   0.6220  0.0153    0.1576   0.0050      141.7474
Δ R²: +0.0187
Δ MAE: -0.0043

LEVEL 3
                  model  mean_r2  std_r2  mean_mae  std_mae  median_alpha
               Baseline   0.6018  0.0312    0.1531   0.0104       93.2603
Field-size interactions   0.6210  0.0348    0.1487   0.0106      107.2267
Δ R²: +0.0193
Δ MAE: -0.0044

LEVEL 4
                  model  mean_r2  std_r2  mean_mae  std_mae  median_alpha
               Baseline   0.5755  0.0368    0.1488   0.0083       93.2603
Field-size interac

In [34]:
field_interaction_coefficients = []

for level in LEVEL_ORDER:
    level_df = interaction_model_samples[
        level
    ]

    features = interaction_model_features[
        level
    ]

    predictor_cols = (
        features["main"]
        + ["log_field_size_centered"]
        + features["interactions"]
    )

    X = level_df[predictor_cols]
    y = level_df["placement_pct"]
    groups = level_df["competition_id"]

    outer_cv = GroupKFold(
        n_splits=N_SPLITS
    )

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        outer_cv.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        train_groups = groups.iloc[
            train_idx
        ]

        inner_cv = GroupKFold(
            n_splits=N_SPLITS
        )

        inner_splits = list(
            inner_cv.split(
                X_train,
                y_train,
                groups=train_groups,
            )
        )

        model = RidgeCV(
            alphas=RIDGE_ALPHAS,
            cv=inner_splits,
        )

        model.fit(
            X_train,
            y_train,
        )

        coefficient_map = dict(
            zip(
                predictor_cols,
                model.coef_,
            )
        )

        for category, interaction_col in zip(
            [
                col.replace("_z", "")
                for col in features["main"]
            ],
            features["interactions"],
        ):
            coefficient = (
                coefficient_map[
                    interaction_col
                ]
            )

            field_interaction_coefficients.append({
                "level": level,
                "fold": fold,
                "category": category,
                "coefficient": coefficient,
            })

field_interaction_coefficients = pd.DataFrame(
    field_interaction_coefficients
)

field_interaction_stability = (
    field_interaction_coefficients
    .groupby(
        ["level", "category"],
        as_index=False,
    )
    .agg(
        mean_coefficient=(
            "coefficient",
            "mean",
        ),
        std_coefficient=(
            "coefficient",
            "std",
        ),
        min_coefficient=(
            "coefficient",
            "min",
        ),
        max_coefficient=(
            "coefficient",
            "max",
        ),
        negative_folds=(
            "coefficient",
            lambda x: (x < 0).sum(),
        ),
        positive_folds=(
            "coefficient",
            lambda x: (x > 0).sum(),
        ),
    )
)

field_interaction_stability[
    "abs_mean_coefficient"
] = (
    field_interaction_stability[
        "mean_coefficient"
    ].abs()
)

print(
    "FIELD-SIZE INTERACTION "
    "COEFFICIENT STABILITY"
)

print(
    "Negative = category relationship "
    "strengthens as field size increases"
)

print(
    "Positive = category relationship "
    "weakens as field size increases"
)

for level in LEVEL_ORDER:
    print(f"\n{'=' * 70}")
    print(f"LEVEL {level}")
    print("=" * 70)

    results = (
        field_interaction_stability[
            field_interaction_stability[
                "level"
            ] == level
        ]
        .sort_values(
            "abs_mean_coefficient",
            ascending=False,
        )
    )

    print(
        results.to_string(
            index=False,
            columns=[
                "category",
                "mean_coefficient",
                "std_coefficient",
                "min_coefficient",
                "max_coefficient",
                "negative_folds",
                "positive_folds",
            ],
        )
    )

FIELD-SIZE INTERACTION COEFFICIENT STABILITY
Negative = category relationship strengthens as field size increases
Positive = category relationship weakens as field size increases

LEVEL 1
                   category  mean_coefficient  std_coefficient  min_coefficient  max_coefficient  negative_folds  positive_folds
                         rc            0.0300           0.0029           0.0268           0.0346               0               5
         pyramid_difficulty            0.0261           0.0026           0.0221           0.0288               0               5
 running_tumbling_execution            0.0229           0.0053           0.0180           0.0316               0               5
standing_tumbling_execution            0.0197           0.0040           0.0159           0.0246               0               5
            stunt_execution            0.0192           0.0035           0.0153           0.0246               0               5
                       show           

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

scaled_field_interaction_results = []
scaled_field_interaction_coefficients = []

for level in LEVEL_ORDER:
    level_df = interaction_model_samples[
        level
    ]

    features = interaction_model_features[
        level
    ]

    baseline_features = (
        features["main"]
        + ["log_field_size_centered"]
    )

    full_features = (
        baseline_features
        + features["interactions"]
    )

    X_baseline = level_df[
        baseline_features
    ]

    X_full = level_df[
        full_features
    ]

    y = level_df["placement_pct"]
    groups = level_df["competition_id"]

    outer_cv = GroupKFold(
        n_splits=N_SPLITS
    )

    for fold, (
        train_idx,
        test_idx,
    ) in enumerate(
        outer_cv.split(
            X_full,
            y,
            groups=groups,
        ),
        start=1,
    ):
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        train_groups = groups.iloc[
            train_idx
        ]

        inner_cv = GroupKFold(
            n_splits=N_SPLITS
        )

        inner_splits = list(
            inner_cv.split(
                X_full.iloc[train_idx],
                y_train,
                groups=train_groups,
            )
        )

        fitted_models = {}

        for model_name, X in [
            (
                "Baseline",
                X_baseline,
            ),
            (
                "Field-size interactions",
                X_full,
            ),
        ]:
            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            pipeline = Pipeline([
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "ridge",
                    Ridge(),
                ),
            ])

            search = GridSearchCV(
                estimator=pipeline,
                param_grid={
                    "ridge__alpha":
                        RIDGE_ALPHAS
                },
                cv=inner_splits,
                scoring="neg_mean_squared_error",
            )

            search.fit(
                X_train,
                y_train,
            )

            predictions = (
                search.predict(
                    X_test
                )
            )

            scaled_field_interaction_results.append({
                "level": level,
                "fold": fold,
                "model": model_name,
                "alpha": (
                    search.best_params_[
                        "ridge__alpha"
                    ]
                ),
                "r2": r2_score(
                    y_test,
                    predictions,
                ),
                "mae": mean_absolute_error(
                    y_test,
                    predictions,
                ),
            })

            fitted_models[
                model_name
            ] = search.best_estimator_

        full_model = fitted_models[
            "Field-size interactions"
        ]

        ridge = full_model.named_steps[
            "ridge"
        ]

        coefficient_map = dict(
            zip(
                full_features,
                ridge.coef_,
            )
        )

        for category, interaction_col in zip(
            [
                col.replace("_z", "")
                for col in features["main"]
            ],
            features["interactions"],
        ):
            scaled_field_interaction_coefficients.append({
                "level": level,
                "fold": fold,
                "category": category,
                "coefficient": (
                    coefficient_map[
                        interaction_col
                    ]
                ),
            })

scaled_field_interaction_results = (
    pd.DataFrame(
        scaled_field_interaction_results
    )
)

scaled_field_interaction_coefficients = (
    pd.DataFrame(
        scaled_field_interaction_coefficients
    )
)

scaled_comparison_summary = (
    scaled_field_interaction_results
    .groupby(
        ["level", "model"],
        as_index=False,
    )
    .agg(
        mean_r2=("r2", "mean"),
        std_r2=("r2", "std"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        median_alpha=("alpha", "median"),
    )
)

scaled_coefficient_stability = (
    scaled_field_interaction_coefficients
    .groupby(
        ["level", "category"],
        as_index=False,
    )
    .agg(
        mean_coefficient=(
            "coefficient",
            "mean",
        ),
        std_coefficient=(
            "coefficient",
            "std",
        ),
        min_coefficient=(
            "coefficient",
            "min",
        ),
        max_coefficient=(
            "coefficient",
            "max",
        ),
        negative_folds=(
            "coefficient",
            lambda x: (x < 0).sum(),
        ),
        positive_folds=(
            "coefficient",
            lambda x: (x > 0).sum(),
        ),
    )
)

scaled_coefficient_stability[
    "abs_mean_coefficient"
] = (
    scaled_coefficient_stability[
        "mean_coefficient"
    ].abs()
)

print(
    "SCALED FIELD-SIZE "
    "INTERACTION MODEL"
)

for level in LEVEL_ORDER:
    level_results = (
        scaled_comparison_summary[
            scaled_comparison_summary[
                "level"
            ] == level
        ]
    )

    baseline = level_results[
        level_results["model"]
        == "Baseline"
    ].iloc[0]

    interaction = level_results[
        level_results["model"]
        == "Field-size interactions"
    ].iloc[0]

    print(f"\nLEVEL {level}")

    print(
        level_results.to_string(
            index=False,
            columns=[
                "model",
                "mean_r2",
                "std_r2",
                "mean_mae",
                "std_mae",
                "median_alpha",
            ],
        )
    )

    print(
        f"Δ R²: "
        f"{interaction['mean_r2'] - baseline['mean_r2']:+.4f}"
    )

    print(
        f"Δ MAE: "
        f"{interaction['mean_mae'] - baseline['mean_mae']:+.4f}"
    )

    print()
    print(
        "Interaction coefficients:"
    )

    coefficients = (
        scaled_coefficient_stability[
            scaled_coefficient_stability[
                "level"
            ] == level
        ]
        .sort_values(
            "abs_mean_coefficient",
            ascending=False,
        )
    )

    print(
        coefficients.to_string(
            index=False,
            columns=[
                "category",
                "mean_coefficient",
                "std_coefficient",
                "min_coefficient",
                "max_coefficient",
                "negative_folds",
                "positive_folds",
            ],
        )
    )

SCALED FIELD-SIZE INTERACTION MODEL

LEVEL 1
                  model  mean_r2  std_r2  mean_mae  std_mae  median_alpha
               Baseline   0.6361  0.0173    0.1589   0.0063      123.2847
Field-size interactions   0.6536  0.0165    0.1544   0.0059      187.3817
Δ R²: +0.0175
Δ MAE: -0.0046

Interaction coefficients:
                   category  mean_coefficient  std_coefficient  min_coefficient  max_coefficient  negative_folds  positive_folds
                         rc            0.0136           0.0015           0.0118           0.0152               0               5
         pyramid_difficulty            0.0113           0.0016           0.0089           0.0133               0               5
 running_tumbling_execution            0.0099           0.0025           0.0068           0.0136               0               5
standing_tumbling_execution            0.0080           0.0023           0.0053           0.0110               0               5
            stunt_execution     

### Field-Size Findings

Competitive field size provides meaningful context for scoring relationships,
particularly in Levels 1–4.

Descriptively, univariate category-placement correlations generally become
stronger in larger fields. However, multivariable competition-held-out models
show that this pattern does not mean every individual scoring category becomes
more independently important as field size increases.

After accounting for correlations among scoring components, adding category-by-
field-size interactions improves out-of-sample placement prediction consistently
in Levels 1–4. Mean held-out R² increases by approximately 0.018–0.019 across
these levels, with mean absolute error also decreasing in every case.

Most stable interaction coefficients in Levels 1–4 are positive. Because lower
normalized placement represents better competitive outcomes, this indicates that
the unique relationship between many individual categories and placement becomes
weaker as field size increases, even though their univariate relationships often
become stronger.

This combination suggests that larger competitive fields may increase the
importance of overall routine quality while reducing the amount of unique
placement information attributable to many individual scoring components once
the rest of the scorecard is considered.

Toss execution is a notable exception. In Levels 3 and 4, its interaction with
field size is negative across all five competition-held-out folds, indicating
that toss execution becomes more independently associated with placement as
competitive fields grow.

Evidence for field-size interactions is substantially weaker in Levels 4.2 and
5, where interaction models do not improve held-out R². Level 6 shows some
predictive improvement from interactions, but the small sample and high
competition-to-competition variability make those relationships exploratory.

These results should not be interpreted as evidence that judges intentionally
change category weighting based on field size. Field size may also capture
differences in competitive depth, event composition, team quality, or the
statistical resolution of normalized placement.

## Cross-Level Synthesis

The preceding analyses evaluate competitive outcomes from several complementary
perspectives: univariate category relationships, multivariable scoring models,
deduction effects, and competitive field size.

This section consolidates those results to identify which findings are
consistent across competitive levels, which relationships change as athletic
difficulty increases, and which results should be treated as level-specific or
exploratory.

The objective is not to introduce additional modeling, but to distinguish the
most reproducible competitive patterns from results that appear only within
individual levels or analytical specifications.

In [40]:
cross_level_summary = []

for level in LEVEL_ORDER:
    level_results = (
        coefficient_stability[
            coefficient_stability["level"] == level
        ]
        .groupby("category", as_index=False)["coefficient"]
        .mean()
        .sort_values("coefficient")
        .reset_index(drop=True)
    )

    top_categories = level_results.head(5)["category"].tolist()

    cross_level_summary.append({
        "level": level,
        "top_1": top_categories[0],
        "top_2": top_categories[1],
        "top_3": top_categories[2],
        "top_4": top_categories[3],
        "top_5": top_categories[4],
    })

cross_level_summary = pd.DataFrame(cross_level_summary)

print("TOP CONDITIONAL SCORING RELATIONSHIPS BY LEVEL")
print(cross_level_summary.to_string(index=False))

TOP CONDITIONAL SCORING RELATIONSHIPS BY LEVEL
level                      top_1              top_2              top_3                       top_4                       top_5
    1         pyramid_difficulty    stunt_execution                 rc                        show standing_tumbling_execution
    2                         rc    stunt_execution pyramid_difficulty standing_tumbling_execution           pyramid_execution
    3 running_tumbling_execution pyramid_difficulty                 rc             stunt_execution           pyramid_execution
    4                       show pyramid_difficulty  pyramid_execution             stunt_execution standing_tumbling_execution
  4.2            stunt_execution pyramid_difficulty                 rc           pyramid_execution                        show
    5         pyramid_difficulty    stunt_execution               show standing_tumbling_execution           pyramid_execution
    6 running_tumbling_execution     toss_execution             

In [41]:
top_five_frequency = (
    cross_level_summary
    .melt(
        id_vars="level",
        value_vars=["top_1", "top_2", "top_3", "top_4", "top_5"],
        value_name="category",
    )
    .groupby("category")["level"]
    .nunique()
    .sort_values(ascending=False)
    .rename("levels_in_top_five")
    .reset_index()
)

print("CATEGORY APPEARANCES IN TOP FIVE ACROSS LEVELS")
print(top_five_frequency.to_string(index=False))

CATEGORY APPEARANCES IN TOP FIVE ACROSS LEVELS
                   category  levels_in_top_five
            stunt_execution                   7
         pyramid_difficulty                   6
                       show                   5
          pyramid_execution                   5
standing_tumbling_execution                   4
                         rc                   4
 running_tumbling_execution                   2
     formations_transitions                   1
             toss_execution                   1


### Cross-Level Category Consistency

Stunt execution appeared among the five most negative average conditional
coefficients at all seven competitive levels. Pyramid difficulty appeared at
six levels, while show and pyramid execution each appeared at five.

These recurring relationships suggest that building-related scores and
overall presentation provide consistent information for distinguishing
placements across levels. However, top-five frequency is a descriptive
summary of model coefficients, not a measure of causal impact or an official
ranking of scoring categories.

The patterns also vary by level. Running tumbling execution appeared in the
top five at Levels 3 and 6, while toss execution and formations/transitions
appeared only at Level 6. Level 6 findings warrant particular caution because
its placement model showed substantially more variability across held-out
competitions.

Because placement is derived from total scores, these results describe which
recorded category scores are most strongly associated with differences in
placement after accounting for other modeled categories. They do not establish
that judges intentionally prioritize those categories or that improving one
category alone would produce the modeled placement change.

### Cross-Level Predictive Performance

Category consistency describes which scoring relationships recur across levels.
Predictive performance addresses a different question: how well do those
relationships generalize to competitions excluded from model training?

The comparison below uses competition-held-out results rather than training
performance. Higher held-out R² indicates that the model explains more
variation in normalized placement in unseen competitions. Variability across
folds is also important, particularly for levels with fewer performances.

Across Levels 1–4, the categories-only models achieved mean held-out R²
values of approximately 0.57–0.64. Performance was lower at Level 5
(R² = 0.50) and Level 6 (R² = 0.30).

Level 4.2 achieved a mean held-out R² of 0.58, although its results varied
across folds. Levels 5 and 6 also showed greater fold-to-fold variability,
with Level 6 particularly unstable.

These results indicate that recorded scoring categories provide useful
information about placement in unseen competitions, but predictive
relationships do not generalize equally well across levels. Findings for
Level 6 should be treated as exploratory rather than as evidence of a
reliable level-wide pattern.

Held-out R² measures predictive performance, not causal impact. Because
placement is derived from total scores, the models are evaluating how well
the distribution of category scores distinguishes competitive outcomes,
rather than predicting an outcome independent of the scoring system.

### Cross-Level Deduction Findings

Across all seven levels, performances with zero recorded deductions had
higher win rates than performances with deductions. Larger deduction totals
were also generally associated with worse normalized placement.

Adding deductions to the category-based placement models improved mean
competition-held-out R² by approximately 0.02–0.03 across levels and reduced
prediction error in every level. This indicates that deductions provide
additional predictive information beyond the recorded category scores.

These findings describe associations, not the causal effect of a penalty.
Deductions may coincide with other performance problems, and the analysis
does not establish what a team's placement would have been had a deduction
not occurred.

### Cross-Level Field-Size Findings

Adding field-size interactions improved competition-held-out placement
prediction at Levels 1–4, increasing mean R² by approximately 0.018–0.019
and reducing prediction error. The interaction models did not provide a
consistent improvement at Levels 4.2 or 5.

Level 6 showed a small improvement in mean R², but its substantial
fold-to-fold variability makes that result exploratory.

Several category relationships changed with field size after accounting
for the other modeled categories. These conditional interaction patterns
sometimes differed from the simpler, one-category-at-a-time relationships,
because scoring categories share predictive information.

The results suggest that competitive field size provides useful context
for interpreting placement relationships, particularly at Levels 1–4.
They do not establish that judges change their scoring priorities based
on the number of teams in a division.

### Synthesis Takeaways

Across levels, stunt execution showed the most consistent conditional
relationship with placement, appearing among the five most negative average
model coefficients at every level. Pyramid difficulty also appeared
frequently, although the ordering and mix of categories varied by level.

Competition-held-out models demonstrated that scoring categories contain
substantial information about placement, particularly at Levels 1–4.
Deductions added predictive information across all levels, while field-size
interactions improved prediction primarily at Levels 1–4.

These findings describe patterns within the recorded scoring system, not
causal effects or judges' intentions. Results for Levels 4.2–6 warrant
additional caution where sample sizes or fold-to-fold performance limit
confidence in generalization.
